# 4.2 FER2013 Trial 001: MobileNetV2 Multiclass Baseline

Purpose:
- Train a multiclass facial expression classifier.
- Use FER2013 seven-class dataset.
- Establish a visual-modality multiclass baseline.

In [1]:
import gc
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from rural_stroke_assist.config import (
    FER2013_SPLIT_MANIFEST_PATH,
    FER2013_IMAGE_SIZE,
    FER2013_BATCH_SIZE,
    FER2013_TRIAL_001_MODEL_PATH,
    FER2013_TRIAL_001_RESULTS_PATH,
    FER2013_TRIAL_001_PREDICTIONS_PATH,
    FER2013_TRIAL_001_CONFUSION_MATRIX_PATH,
    FER2013_TRIAL_001_HISTORY_PATH,
)

from rural_stroke_assist.preprocessing.tf_image_dataset import (
    dataframe_to_multiclass_tf_dataset,
    add_augmentation,
)

from rural_stroke_assist.preprocessing.fer2013 import (
    FER2013_LABELS,
    FER2013_INVERSE_LABEL_MAPPING,
)

from rural_stroke_assist.modeling.vision_models import (
    build_mobilenetv2_multiclass_classifier,
)

from rural_stroke_assist.modeling.callbacks import (
    build_early_stopping_callback,
    build_csv_logger_callback,
)

from rural_stroke_assist.modeling.evaluation import (
    evaluate_multiclass_classifier,
    create_classification_report_dict,
    save_confusion_matrix_plot,
    save_markdown_results,
)

In [2]:
tf.keras.backend.clear_session()
gc.collect()

tf.random.set_seed(42)

print("GPUs:", tf.config.list_physical_devices("GPU"))


GPUs: []


In [3]:
fer_split_df = pd.read_csv(FER2013_SPLIT_MANIFEST_PATH)

fer_split_df.groupby(["split", "class_label"]).size()

split  class_label
test   angry           958
       disgust         111
       fear           1024
       happy          1774
       neutral        1233
       sad            1247
       surprise        831
train  angry          3396
       disgust         371
       fear           3482
       happy          6133
       neutral        4220
       sad            4105
       surprise       2695
val    angry           599
       disgust          65
       fear            615
       happy          1082
       neutral         745
       sad             725
       surprise        476
dtype: int64

In [4]:
train_df = fer_split_df[fer_split_df["split"] == "train"].copy()
val_df = fer_split_df[fer_split_df["split"] == "val"].copy()
test_df = fer_split_df[fer_split_df["split"] == "test"].copy()

len(train_df), len(val_df), len(test_df)

(24402, 4307, 7178)

In [5]:
train_ds = dataframe_to_multiclass_tf_dataset(
    train_df,
    image_size=FER2013_IMAGE_SIZE,
    batch_size=FER2013_BATCH_SIZE,
    shuffle=True,
    seed=42,
)

train_ds = add_augmentation(train_ds)

val_ds = dataframe_to_multiclass_tf_dataset(
    val_df,
    image_size=FER2013_IMAGE_SIZE,
    batch_size=FER2013_BATCH_SIZE,
    shuffle=False,
)

test_ds = dataframe_to_multiclass_tf_dataset(
    test_df,
    image_size=FER2013_IMAGE_SIZE,
    batch_size=FER2013_BATCH_SIZE,
    shuffle=False,
)

In [6]:
for images, labels in train_ds.take(1):
    print(images.shape)
    print(labels.shape)
    print(labels.numpy()[:10])
    print(float(tf.reduce_min(images)), float(tf.reduce_max(images)))

(32, 160, 160, 3)
(32,)
[5 3 3 3 4 0 5 2 2 5]
0.0 1.0


In [7]:
model = build_mobilenetv2_multiclass_classifier(
    image_size=FER2013_IMAGE_SIZE,
    num_classes=len(FER2013_LABELS),
    dropout_rate=0.3,
    learning_rate=1e-4,
)

model.summary()

Model: "mobilenetv2_fer2013_multiclass_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multiply (Multiply)             │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7)              │         8,967 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,266,951 (8.65 MB)

 Trainable params: 8,967 (35.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [8]:
callbacks = [
    build_early_stopping_callback(
        patience=4,
        monitor="val_loss",
    ),
    build_csv_logger_callback(
        output_path=FER2013_TRIAL_001_HISTORY_PATH,
    ),
]

In [9]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
)

Epoch 1/20
763/763 ━━━━━━━━━━━━━━━━━━━━ 302s 391ms/step - accuracy: 0.2591 - loss: 2.0051 - val_accuracy: 0.3638 - val_loss: 1.6510
Epoch 2/20
763/763 ━━━━━━━━━━━━━━━━━━━━ 273s 356ms/step - accuracy: 0.3266 - loss: 1.7844 - val_accuracy: 0.3910 - val_loss: 1.5550
Epoch 3/20
763/763 ━━━━━━━━━━━━━━━━━━━━ 284s 371ms/step - accuracy: 0.3602 - loss: 1.6808 - val_accuracy: 0.4200 - val_loss: 1.5101
Epoch 4/20
763/763 ━━━━━━━━━━━━━━━━━━━━ 289s 378ms/step - accuracy: 0.3882 - loss: 1.6134 - val_accuracy: 0.4279 - val_loss: 1.4792
Epoch 5/20
763/763 ━━━━━━━━━━━━━━━━━━━━ 250s 283ms/step - accuracy: 0.4070 - loss: 1.5650 - val_accuracy: 0.4442 - val_loss: 1.4527
Epoch 6/20
763/763 ━━━━━━━━━━━━━━━━━━━━ 233s 305ms/step - accuracy: 0.4129 - loss: 1.5420 - val_accuracy: 0.4425 - val_loss: 1.4433
Epoch 7/20
763/763 ━━━━━━━━━━━━━━━━━━━━ 222s 291ms/step - accuracy: 0.4205 - loss: 1.5149 - val_accuracy: 0.4521 - val_loss: 1.4269
Epoch 8/20
763/763 ━━━━━━━━━━━━━━━━━━━━ 215s 281ms/step - accuracy: 0.4304 -

In [10]:
raw_probs = model.predict(test_ds)

y_pred = raw_probs.argmax(axis=1)
y_test = test_df["label_encoded"].to_numpy(dtype=int)

np.unique(y_pred, return_counts=True), np.unique(y_test, return_counts=True)

225/225 ━━━━━━━━━━━━━━━━━━━━ 56s 247ms/step


((array([0, 1, 2, 3, 4, 5, 6], dtype=int64),
  array([ 520,    9,  336, 2495, 1603, 1473,  742], dtype=int64)),
 (array([0, 1, 2, 3, 4, 5, 6]),
  array([ 958,  111, 1024, 1774, 1233, 1247,  831], dtype=int64)))

In [11]:
metrics = evaluate_multiclass_classifier(
    y_true=y_test,
    y_pred=y_pred,
)

metrics

{'accuracy': 0.48286430760657567,
 'balanced_accuracy': 0.398032583622828,
 'macro_precision': 0.4917778826652207,
 'macro_recall': 0.398032583622828,
 'macro_f1': 0.39478808974814245,
 'weighted_f1': 0.4564311600861201}

In [12]:
report = create_classification_report_dict(
    y_true=y_test,
    y_pred=y_pred,
    target_names=FER2013_LABELS,
)

report

{'angry': {'precision': 0.4326923076923077,
  'recall': 0.2348643006263048,
  'f1-score': 0.3044654939106901,
  'support': 958.0},
 'disgust': {'precision': 0.5555555555555556,
  'recall': 0.04504504504504504,
  'f1-score': 0.08333333333333333,
  'support': 111.0},
 'fear': {'precision': 0.42857142857142855,
  'recall': 0.140625,
  'f1-score': 0.21176470588235294,
  'support': 1024.0},
 'happy': {'precision': 0.5466933867735471,
  'recall': 0.7688838782412627,
  'f1-score': 0.639025532911689,
  'support': 1774.0},
 'neutral': {'precision': 0.40736119775421087,
  'recall': 0.5296025952960259,
  'f1-score': 0.4605077574047955,
  'support': 1233.0},
 'sad': {'precision': 0.38289205702647655,
  'recall': 0.45228548516439454,
  'f1-score': 0.4147058823529412,
  'support': 1247.0},
 'surprise': {'precision': 0.6886792452830188,
  'recall': 0.614921780986763,
  'f1-score': 0.6497139224411952,
  'support': 831.0},
 'accuracy': 0.48286430760657567,
 'macro avg': {'precision': 0.4917778826652207

In [13]:
save_confusion_matrix_plot(
    y_true=y_test,
    y_pred=y_pred,
    class_names=FER2013_LABELS,
    output_path=FER2013_TRIAL_001_CONFUSION_MATRIX_PATH,
)

FER2013_TRIAL_001_CONFUSION_MATRIX_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/reports/figures/experiments/fer2013_trial_001_mobilenetv2_160_confusion_matrix.png')

In [14]:
FER2013_TRIAL_001_PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)

predictions_df = test_df.copy()
predictions_df["predicted_label_encoded"] = y_pred
predictions_df["predicted_label"] = predictions_df["predicted_label_encoded"].map(
    FER2013_INVERSE_LABEL_MAPPING
)
predictions_df["true_label"] = predictions_df["label_encoded"].map(
    FER2013_INVERSE_LABEL_MAPPING
)
predictions_df["is_correct"] = (
    predictions_df["label_encoded"] == predictions_df["predicted_label_encoded"]
)

for class_index, class_name in FER2013_INVERSE_LABEL_MAPPING.items():
    predictions_df[f"prob_{class_name}"] = raw_probs[:, class_index]

predictions_df.to_csv(FER2013_TRIAL_001_PREDICTIONS_PATH, index=False)

FER2013_TRIAL_001_PREDICTIONS_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/data/processed/experiments/fer2013_trial_001_mobilenetv2_160_predictions.csv')

In [15]:
save_markdown_results(
    metrics=metrics,
    report=report,
    output_path=FER2013_TRIAL_001_RESULTS_PATH,
    title="FER2013 Trial 001 MobileNetV2 Multiclass Results",
)

FER2013_TRIAL_001_RESULTS_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/reports/experiments/fer2013_trial_001_mobilenetv2_160_results.md')

In [16]:
accepted = (
    metrics["accuracy"] >= 0.45
    and metrics["macro_f1"] >= 0.35
)

{
    "accuracy": metrics["accuracy"],
    "macro_f1": metrics["macro_f1"],
    "accepted": accepted,
}

{'accuracy': 0.48286430760657567,
 'macro_f1': 0.39478808974814245,
 'accepted': True}

In [ ]:
if accepted:
    FER2013_TRIAL_001_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    model.save(FER2013_TRIAL_001_MODEL_PATH)
    print("Accepted FER2013 model saved to:", FER2013_TRIAL_001_MODEL_PATH)
else:
    print("Model rejected. Results and predictions saved, model artifact not saved.")

Accepted FER2013 model saved to: C:\Users\Asus\Downloads\Uni_work\Grad-Project\rural-stroke-assist\models\experiments\fer2013\trial_001_mobilenetv2_160\model.keras


: 

## Trial Notes

This is the first multiclass visual baseline.

The goal is not to reach state-of-the-art FER2013 performance immediately. The goal is to validate that the visual pipeline can support multiclass facial-expression recognition with reproducible data handling, training, evaluation, and reporting.